# Data Anonymization for Tabular Data

This notebook demonstrates how to utilize the **READI** framework to perform data anonymization on tabular/structured datasets. Data anonymization is a crucial step in preparing datasets for sharing or secondary analysis by ensuring that individuals represented in the data cannot be re-identified, while still preserving as much utility and statistical value as possible.

We will explore two popular and powerful tabular anonymization algorithms supported by READI:

1. **Mondrian**: A highly efficient top-down multidimensional partitioning algorithm suitable for both numeric and categorical data.
2. **Optimal Lattice Anonymization (OLA)**: A bottom-up lattice exploration algorithm that searches for the globally optimal combination of generalization levels across specified hierarchy trees to minimize information loss.

We will also see how to combine **$k$-Anonymity** with advanced constraints such as **$l$-Diversity** and **$t$-Closeness**.

## 1. Preparing the Example Dataset

We will load the synthetic `healthcare-dataset.csv` file, which represents typical demographic and clinical records for patients. This dataset contains a mix of direct identifiers, quasi-identifiers, and sensitive clinical attributes.

In [1]:
import pandas as pd

# Load the dataset and fill missing values with 'Unknown'
df = pd.read_csv("./healthcare-dataset.csv", header=None).fillna("Unknown")

# Assign readable column names based on the dataset schema
df.columns = [
    "patient_id",
    "name",
    "surname",
    "email",
    "yob",
    "zip_code",
    "gender",
    "ethnicity",
    "religion",
    "marital_status",
    "icd_code",
]

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (32440, 11)


,patient_id,name,surname,email,yob,zip_code,gender,ethnicity,religion,marital_status,icd_code
0,P00001,Sophia,Rosu,RosSophi@hotmail.co.uk,1977,33917,Male,White,Roman Catholic,Never-married,401.9
1,P00002,Kenny,Bogner,Ke@gmail.com,1966,32526,Male,White,Baptist,Married-civ-spouse,244.9
2,P00003,Lawrence,Moneypenny,MoneypLawrence@live.com,1978,32811,Male,White,Unknown,Divorced,530.81
3,P00004,Katelyn,Martsolf,MartKa@gmail.com,1963,34453,Male,Black,Unknown,Married-civ-spouse,250.00
4,P00005,Kolby,Mcglon,McKolby@gmail.com,1988,33596,Female,Black,Unknown,Married-civ-spouse,401.9


### Preprocessing: Direct Identifier Suppression

Direct identifiers (like names, emails, and patient IDs) must be entirely removed or suppressed before anonymizing the remaining attributes, as they can directly pinpoint an individual. Quasi-identifiers (like year of birth, ZIP code, gender, ethnicity, and marital status) will be generalized. Sensitive attributes (like the ICD diagnosis code) will be kept intact but guarded by privacy constraints.

In [2]:
# Drop direct identifiers to create our target anonymization table
target_df = df.drop(columns=["patient_id", "name", "surname", "email"])
target_df.head()

,yob,zip_code,gender,ethnicity,religion,marital_status,icd_code
0,1977,33917,Male,White,Roman Catholic,Never-married,401.9
1,1966,32526,Male,White,Baptist,Married-civ-spouse,244.9
2,1978,32811,Male,White,Unknown,Divorced,530.81
3,1963,34453,Male,Black,Unknown,Married-civ-spouse,250.00
4,1988,33596,Female,Black,Unknown,Married-civ-spouse,401.9


## 2. Multidimensional k-Anonymity using Mondrian

The Mondrian algorithm partitions the multidimensional quasi-identifier space using a top-down greedy KD-tree-like split strategy. For numerical or ordered columns, it generalizes values into ranges (e.g., `1970-1980`).

In this section, we will anonymize the dataset using **$k$-Anonymity ($k = 5$)** over two quasi-identifiers:
- `yob` (Year of Birth) - Numerical
- `zip_code` (ZIP Code) - Numerical

In [3]:
from risk_assessment.anonymization import KAnonymity
from risk_assessment.anonymization.mondrian import Mondrian, MondrianOptions
from risk_assessment.metrics.informationloss import ColumnClass, ColumnInformation, ColumnType
from risk_assessment.utility.hierarchy import NumericalRange

# Define column information for each column in target_df
# Columns: yob, zip_code, gender, ethnicity, religion, marital_status, icd_code
column_info_mondrian = [
    ColumnInformation(ColumnType.QUASI, column_class=ColumnClass.NUMERIC, range=NumericalRange(target_df["yob"])),
    ColumnInformation(ColumnType.QUASI, column_class=ColumnClass.NUMERIC, range=NumericalRange(target_df["zip_code"])),
    ColumnInformation(),  # gender (not anonymizing in this run)
    ColumnInformation(),  # ethnicity (not anonymizing in this run)
    ColumnInformation(),  # religion (not anonymizing in this run)
    ColumnInformation(),  # marital_status (not anonymizing in this run)
    ColumnInformation(ColumnType.SENSITIVE),  # icd_code
]

# Set up Mondrian to enforce k-Anonymity with k=5
mondrian = Mondrian(MondrianOptions([KAnonymity(k=5)]))

# Run the anonymization
anon_mondrian_df, report_mondrian = mondrian.anonymize(target_df, column_info_mondrian)

print("Mondrian Anonymization Complete!")
anon_mondrian_df.head()

Mondrian Anonymization Complete!


,yob,zip_code,gender,ethnicity,religion,marital_status,icd_code
4743,1937-1940,32003-32177,Female,Black,Unknown,Widowed,V70.0
6013,1937-1940,32003-32177,Male,White,Nonreligious,Never-married,244.9
8504,1937-1940,32003-32177,Male,White,Unknown,Married-civ-spouse,V58.69
8994,1937-1940,32003-32177,Male,White,Church of Christ,Married-civ-spouse,401.9
12624,1937-1940,32003-32177,Male,Other,Nonreligious,Married-civ-spouse,250.00


### Verifying k-Anonymity

Let's verify that every unique group of quasi-identifiers in our anonymized dataset has a frequency of at least $k = 5$.

In [4]:
# Group by the quasi-identifiers and check minimum group size
group_sizes = anon_mondrian_df.groupby(["yob", "zip_code"]).size()
print(f"Minimum equivalence class size: {group_sizes.min()}")
print(f"Total unique equivalence classes: {len(group_sizes)}")
assert group_sizes.min() >= 5

Minimum equivalence class size: 5
Total unique equivalence classes: 4528


## 3. Optimal Lattice Anonymization (OLA)

Unlike partition-based methods, OLA performs global generalization based on pre-defined hierarchy trees. It searches the entire generalization space (modeled as a lattice) and identifies the globally optimal generalization levels that minimize overall information loss while ensuring all privacy constraints are met.

Let's construct generalization hierarchies for our categorical and numerical quasi-identifiers.

In [5]:
from risk_assessment.utility.hierarchy import MaterializedHierarchy
from risk_assessment.utility.hierarchy.datatypes.yob import YOBHierarchy

# 1. Create a Year of Birth Hierarchy programmatically using READI's built-in generator
yob_hierarchy = YOBHierarchy()

# 2. Create custom categorical hierarchies using MaterializedHierarchy
gender_hierarchy = MaterializedHierarchy(
    [["Male", "Person", "*"], ["Female", "Person", "*"], ["Unknown", "Person", "*"]]
)

ethnicity_hierarchy = MaterializedHierarchy(
    [
        ["White", "Non-minority", "*"],
        ["Black", "Minority", "*"],
        ["Asian-Pac-Islander", "Minority", "*"],
        ["Amer-Indian-Eskimo", "Minority", "*"],
        ["Other", "Minority", "*"],
        ["Unknown", "Unknown", "*"],
    ]
)

marital_status_hierarchy = MaterializedHierarchy(
    [
        ["Never-married", "Single", "*"],
        ["Divorced", "Separated", "*"],
        ["Separated", "Separated", "*"],
        ["Widowed", "Single", "*"],
        ["Married-civ-spouse", "Married", "*"],
        ["Married-spouse-absent", "Separated", "*"],
        ["Unknown", "Unknown", "*"],
    ]
)

### Configuring and Running OLA

We will set up OLA to anonymize a 4-dimensional quasi-identifier set: `yob`, `gender`, `ethnicity`, and `marital_status`.

In [6]:
from risk_assessment.anonymization.optimal_lattice_anonymization import OLA, OLAOptions

# Define column information with generalization hierarchies
# Columns: yob, zip_code, gender, ethnicity, religion, marital_status, icd_code
column_info_ola = [
    ColumnInformation(ColumnType.QUASI, column_class=ColumnClass.CATEGORICAL, hierarchy=yob_hierarchy),
    ColumnInformation(),  # zip_code
    ColumnInformation(ColumnType.QUASI, column_class=ColumnClass.CATEGORICAL, hierarchy=gender_hierarchy),
    ColumnInformation(ColumnType.QUASI, column_class=ColumnClass.CATEGORICAL, hierarchy=ethnicity_hierarchy),
    ColumnInformation(),  # religion
    ColumnInformation(ColumnType.QUASI, column_class=ColumnClass.CATEGORICAL, hierarchy=marital_status_hierarchy),
    ColumnInformation(ColumnType.SENSITIVE),  # icd_code
]

# Set OLA Options to enforce k-Anonymity (k=5)
# Note: The second parameter is the suppression threshold (in percent, e.g., 10% maximum record suppression allowed)
ola_options = OLAOptions([KAnonymity(k=5)], suppression=10.0)

ola = OLA(ola_options)

# Run OLA anonymization
anon_ola_df, report_ola = ola.anonymize(target_df, column_info_ola)

print("OLA Anonymization Complete!")
print(f"Suppression rate: {report_ola.suppression_rate:.2f}%")
print(f"Optimal generalization levels: {report_ola.generalization_levels}")
anon_ola_df.head()

TypeError: unhashable type: 'Series'

## 4. Enforcing Advanced Constraints (l-Diversity)

While $k$-Anonymity prevents identity disclosure, it is vulnerable to attribute disclosure if all individuals in an equivalence class have the same sensitive attribute value (e.g., they all have the same diagnosis code). To mitigate this, **$l$-Diversity** ensures that each equivalence class contains at least $l$ distinct values for the sensitive attribute.

Let's configure OLA with **$k$-Anonymity ($k=5$)** and **Distinct $l$-Diversity ($l=2$)** on the sensitive `icd_code` attribute.

In [ ]:
from risk_assessment.anonymization import DistinctLDiversity

# Configure OLA with both k-Anonymity and Distinct l-Diversity
advanced_ola_options = OLAOptions([KAnonymity(k=5), DistinctLDiversity(l=2)], suppression=10.0)

advanced_ola = OLA(advanced_ola_options)

# Run the anonymization
anon_advanced_df, report_advanced = advanced_ola.anonymize(target_df, column_info_ola)

print("Advanced OLA Complete!")
print(f"Suppression rate: {report_advanced.suppression_rate:.2f}%")
print(f"Optimal generalization levels: {report_advanced.generalization_levels}")
anon_advanced_df.head()

## Summary

In this notebook, we learned how to:
1. **Preprocess and prepare** tabular datasets for anonymization using READI.
2. Use **Mondrian** for extremely fast partition-based numerical range generalization.
3. Use **Optimal Lattice Anonymization (OLA)** for global bottom-up optimal generalization with custom categorical trees.
4. Enforce **$k$-Anonymity** and enhance it with **$l$-Diversity** to robustly guard against both identity and attribute disclosure.